# AI–Quantum Finance Portfolio Optimization — Google Colab

Notebook này tái lập pipeline **Data 17/8** đã commit: dữ liệu HOSE complete-case → đặc trưng → XGBoost/EWMA → adaptive universe reduction → cardinality-constrained QUBO → Exact/SA/Penalty-QAOA/XY-QAOA → tối ưu tỷ trọng cổ điển → walk-forward backtest → kiểm định H1–H6.

Mặc định notebook chạy lại đủ 33 folds. Artifact đã công bố cũng được nạp trước để luôn có một mốc đối chiếu. Nghiên cứu mang tính **exploratory**, XY-QAOA chạy trên ideal statevector simulator và không phải bằng chứng quantum advantage hay khuyến nghị đầu tư.

In [ ]:
# CẤU HÌNH NGƯỜI DÙNG
from pathlib import Path

REPO_URL = 'https://github.com/23022006muki/AI-Quantum---Finance-Portfolio-Optimization.git'
SOURCE_COMMIT = 'b672fc4531d9bab8750f06a2d426c3eb62252a7a'
RUN_TESTS = True
RUN_FULL_PIPELINE = True   # True: train/backtest lại đủ 33 folds; False: chỉ đọc kết quả đã công bố
REPO_DIR = Path('/content/AI-Quantum-Finance-Portfolio-Optimization')
PROJECT_DIR = REPO_DIR / 'quantum_portfolio_data'
EXPERIMENT_ID = '20260820T160429-4f2cfc123d'
print({'commit': SOURCE_COMMIT, 'run_tests': RUN_TESTS, 'run_full_pipeline': RUN_FULL_PIPELINE})

## 1. Clone đúng bản đã commit và kiểm tra phiên bản
Notebook không chạy trực tiếp trên nhánh thay đổi liên tục mà checkout commit chứa mã nguồn, gói dữ liệu tái lập và artifact chuẩn.

In [ ]:
import os, shutil, subprocess, sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
print('Repository verified:', actual_commit)

## 2. Cài toàn bộ thư viện
Các phiên bản dưới đây trùng với `pyproject.toml`; `pytest` được cài để kiểm tra hệ thống trước khi chạy mô hình.

In [ ]:
PINNED_PACKAGES = [
    'numpy==2.2.6', 'pandas==2.3.1', 'pyarrow==24.0.0',
    'scikit-learn==1.8.0', 'scipy==1.16.1', 'xgboost==3.3.0',
    'matplotlib==3.10.5', 'PyYAML==6.0.2', 'requests==2.32.5',
    'vnstock==4.0.5', 'finance-datareader==0.9.202', 'pip-system-certs==5.3',
    'streamlit==1.58.0', 'pypdf==6.1.3', 'pdfplumber==0.11.7',
    'Pillow==11.3.0', 'pytest==9.1.1'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *PINNED_PACKAGES], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
print('Installed', len(PINNED_PACKAGES), 'pinned packages and the local project.')

## 3. Khôi phục dữ liệu và kết quả đã công bố
Gói runtime chứa panel giá 120 mã, benchmark VNAllShare TRI, security master, corporate actions, universe và audit. Gói experiment chứa đủ 54 artifact của lần chạy chuẩn. SHA-256 được kiểm tra trước khi giải nén.

In [ ]:
import hashlib, json, zipfile

BUNDLE_DIR = PROJECT_DIR / 'colab_bundle'
WORKSPACE = PROJECT_DIR / 'outputs' / 'Data 17_8'
PUBLISHED_DIR = WORKSPACE / 'outputs' / 'experiments' / EXPERIMENT_ID
EXPECTED_HASHES = {
    'data_17_8_runtime.zip': '4db33cf993977a9e55baf9e7c12ea52a4bc7adb70fe7fd8d29c4ff7188908f30',
    'data_17_8_published_experiment.zip': '5d5faefa264f18647ab86501664013d593a8038b222637f477ad586f52aae5ec',
}
def sha256(path, block=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(block), b''):
            digest.update(chunk)
    return digest.hexdigest()

for filename, expected in EXPECTED_HASHES.items():
    archive = BUNDLE_DIR / filename
    actual = sha256(archive)
    assert actual == expected, f'Checksum mismatch: {filename}'
    print('SHA-256 verified:', filename, actual)

WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_DIR / 'data_17_8_runtime.zip') as zf:
    zf.extractall(WORKSPACE)
PUBLISHED_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_DIR / 'data_17_8_published_experiment.zip') as zf:
    zf.extractall(PUBLISHED_DIR)
assert (PUBLISHED_DIR / 'manifest.json').exists()
print('Runtime workspace:', WORKSPACE)
print('Published experiment:', PUBLISHED_DIR)

## 4. Kiểm thử mã nguồn
Bước này xác nhận các module dữ liệu, walk-forward, QUBO, solver, chi phí và audit vẫn vượt qua test suite trên môi trường Colab.

In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT_DIR, check=True)
else:
    print('RUN_TESTS=False — skipped by user.')

## 5. Mô tả hệ thống và dữ liệu nghiên cứu

Luồng xử lý là: **HOSE point-in-time/complete-case panel → feature engineering trong từng training window → XGBoost ranking và EWMA covariance → adaptive universe reduction → chọn 4/8 tài sản bằng QUBO → XY-QAOA/Dicke và solver đối chứng → classical constrained weighting → monthly walk-forward backtest sau chi phí → block bootstrap và Holm correction**.

XGBoost tạo thứ hạng tương đối chứ không dự báo chính xác mức giá. EWMA trong hệ thống chính là ước lượng hiệp phương sai đa biến. XY-QAOA bảo toàn Hamming weight trên simulator lý tưởng; tỷ trọng liên tục vẫn được giải bằng tối ưu cổ điển.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image

pd.set_option('display.max_columns', 100)
prices = pd.read_parquet(WORKSPACE / 'outputs' / 'normalized' / 'prices.parquet')
benchmark = pd.read_parquet(WORKSPACE / 'outputs' / 'normalized' / 'benchmark.parquet')
master = pd.read_parquet(WORKSPACE / 'outputs' / 'normalized' / 'security_master.parquet')
full_master = pd.read_parquet(WORKSPACE / 'outputs' / 'normalized' / 'security_master_full.parquet')
actions = pd.read_parquet(WORKSPACE / 'outputs' / 'normalized' / 'corporate_actions.parquet')
data_audit = json.loads((WORKSPACE / 'outputs' / 'reports' / 'DATA_17_8_AUDIT.json').read_text(encoding='utf-8'))
published_quality = json.loads((PUBLISHED_DIR / 'data_quality.json').read_text(encoding='utf-8'))
published_leakage = json.loads((PUBLISHED_DIR / 'leakage_audit.json').read_text(encoding='utf-8'))

date_col = 'date' if 'date' in prices.columns else 'trading_date'
ticker_col = 'ticker' if 'ticker' in prices.columns else 'symbol'
summary = pd.DataFrame({
    'Chỉ tiêu': ['Số quan sát', 'Số cổ phiếu runtime', 'Số mã security master đầy đủ', 'Ngày bắt đầu', 'Ngày kết thúc', 'Số phiên benchmark', 'Số sự kiện doanh nghiệp', 'Data-quality gate', 'Leakage audit'],
    'Kết quả': [len(prices), prices[ticker_col].nunique(), len(full_master), str(pd.to_datetime(prices[date_col]).min().date()), str(pd.to_datetime(prices[date_col]).max().date()), len(benchmark), len(actions), published_quality.get('status'), published_leakage.get('status')]
})
display(summary)
display(prices.head())
display(Markdown('**Diễn giải:** panel dùng để huấn luyện có 120 mã đạt complete-case gate, không đồng nghĩa toàn bộ HOSE. Dữ liệu tài chính PIT và ngành lịch sử chưa đủ nên kết quả chỉ mang tính khám phá.'))

## 6. Chạy lại toàn bộ pipeline
Cell này có thể cần thời gian dài trên Colab CPU vì thực hiện lại feature engineering, huấn luyện, 33 folds, bốn nhóm solver, ablation, sensitivity và bootstrap. Đổi `RUN_FULL_PIPELINE=False` ở đầu notebook nếu chỉ cần kiểm tra artifact chuẩn.

In [ ]:
experiments_root = WORKSPACE / 'outputs' / 'experiments'
before = {p.resolve() for p in experiments_root.iterdir() if p.is_dir()}
if RUN_FULL_PIPELINE:
    command = [sys.executable, '-m', 'src.cli', 'run-data-17-8', '--config', 'configs/data_17_8.yaml']
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    after = {p.resolve() for p in experiments_root.iterdir() if p.is_dir()}
    created = sorted(after - before, key=lambda p: p.stat().st_mtime)
    ACTIVE_EXPERIMENT = created[-1] if created else max(after, key=lambda p: p.stat().st_mtime)
else:
    ACTIVE_EXPERIMENT = PUBLISHED_DIR.resolve()
print('ACTIVE_EXPERIMENT =', ACTIVE_EXPERIMENT)

## 7. Kết quả tín hiệu, AUR và bộ giải
Các bảng sau được đọc từ experiment đang hoạt động: run mới nếu `RUN_FULL_PIPELINE=True`, ngược lại là artifact chuẩn.

In [ ]:
manifest = json.loads((ACTIVE_EXPERIMENT / 'manifest.json').read_text(encoding='utf-8'))
rankings = pd.read_csv(ACTIVE_EXPERIMENT / 'rankings.csv')
comparisons = pd.read_csv(ACTIVE_EXPERIMENT / 'comparisons.csv')
tests = pd.read_csv(ACTIVE_EXPERIMENT / 'statistical_tests.csv')
aur = pd.read_csv(ACTIVE_EXPERIMENT / 'aur_diagnostics.csv')

rank_by_fold = rankings.groupby('fold')[['xgboost_rank_ic', 'ewma_rank_ic']].first()
signal_table = pd.DataFrame({
    'Chỉ tiêu': ['XGBoost mean Rank IC', 'XGBoost median Rank IC', 'EWMA mean Rank IC', 'XGBoost − EWMA', 'Holm-adjusted p-value'],
    'Giá trị': [rank_by_fold.xgboost_rank_ic.mean(), rank_by_fold.xgboost_rank_ic.median(), rank_by_fold.ewma_rank_ic.mean(), tests.loc[tests.test.eq('xgboost_rank_ic_vs_ewma_rank_ic'), 'mean_difference'].iloc[0], tests.loc[tests.test.eq('xgboost_rank_ic_vs_ewma_rank_ic'), 'p_value_holm'].iloc[0]]
})
display(Markdown(f"**Experiment:** `{manifest['experiment_id']}` — {manifest['folds_completed']}/{manifest['folds_requested']} folds; OOS {manifest['actual_oos_start']} đến {manifest['actual_oos_end']}."))
display(signal_table.style.format({'Giá trị': '{:.6f}'}))
display(tests[tests.hypothesis.isin(['H1','H2'])][['test','mean_difference','ci_low','ci_high','p_value_holm','conclusion']])
solver_labels = {
    'exact': 'Exact', 'simulated_annealing': 'Simulated Annealing',
    'penalty_stochastic_baseline': 'Penalty stochastic baseline',
    'penalty_qaoa_ideal_statevector': 'Penalty-QAOA',
    'xy_qaoa_dicke_ideal_statevector': 'XY-QAOA + Dicke'
}
solver_table = comparisons.copy()
solver_table['Bộ giải'] = solver_table.method.map(solver_labels).fillna(solver_table.method)
solver_table = solver_table[['Bộ giải','runs','feasibility_rate','optimality_gap_mean','energy_mean','runtime_seconds']]
display(solver_table.style.format({'feasibility_rate':'{:.2%}','optimality_gap_mean':'{:.2%}','energy_mean':'{:.6f}','runtime_seconds':'{:.4f}'}))
display(Markdown('XY-QAOA đạt feasibility 100% và gap thấp hơn Penalty-QAOA trong simulator lý tưởng. Exact và Simulated Annealing vẫn đạt gap 0 trong bài toán nhỏ; do đó kết quả không chứng minh ưu thế lượng tử.'))

## 8. Hiệu quả danh mục ngoài mẫu và rổ cuối
Lợi nhuận được trình bày sau chi phí giao dịch. Sharpe sử dụng lãi suất phi rủi ro 3%/năm theo cấu hình nghiên cứu.

In [ ]:
metrics = pd.read_csv(ACTIVE_EXPERIMENT / 'strategy_metrics_summary.csv')
preferred = ['full_pipeline_xy_qaoa','benchmark_vnallsharetri','liquidity_topk_exact','minimum_variance','equal_weight_universe','ewma_topk_exact','adaptive_exact','xgboost_topk_exact','xgboost_penalty_qaoa']
shown = metrics[metrics.strategy.isin(preferred)].copy()
shown['order'] = shown.strategy.map({name:i for i,name in enumerate(preferred)})
shown = shown.sort_values('order')[['strategy','cumulative_return','annualized_return','annualized_volatility','sharpe','max_drawdown','turnover','total_cost']]
display(shown.style.format({'cumulative_return':'{:.2%}','annualized_return':'{:.2%}','annualized_volatility':'{:.2%}','sharpe':'{:.4f}','max_drawdown':'{:.2%}','turnover':'{:.2f}','total_cost':'{:.2%}'}))

for figure in ['equity_curve.png', 'drawdown.png', 'risk_return.png']:
    path = ACTIVE_EXPERIMENT / 'figures' / figure
    if path.exists(): display(Image(filename=str(path)))

latest = pd.read_csv(ACTIVE_EXPERIMENT / 'latest_selected_portfolio.csv')
latest_summary = json.loads((ACTIVE_EXPERIMENT / 'latest_portfolio_summary.json').read_text(encoding='utf-8'))
basket = latest[['ticker','company_name','target_weight','adv_participation','selection_reason']].copy()
cash_weight = latest_summary.get('cash_weight', 1.0 - basket.target_weight.sum())
cash_row = pd.DataFrame([{'ticker':'CASH','company_name':'Tiền mặt','target_weight':cash_weight,'adv_participation':np.nan,'selection_reason':'market-regime exposure overlay'}])
basket = pd.concat([basket, cash_row], ignore_index=True)
display(Markdown(f"**Rổ cuối tại ngày quyết định {latest.decision_time.iloc[0]}**"))
display(basket.style.format({'target_weight':'{:.2%}','adv_participation':'{:.4%}'}, na_rep='—'))

## 9. Kết luận kiểm định giả thuyết
Notebook diễn giải từng giả thuyết theo đúng phạm vi kiểm định, không dùng quy tắc “chỉ cần một kiểm định thành phần có ý nghĩa là hỗ trợ toàn bộ giả thuyết”.

In [ ]:
def test_row(name):
    return tests.loc[tests.test.eq(name)].iloc[0]

h1 = test_row('xgboost_rank_ic_vs_ewma_rank_ic')
h2_return = test_row('adaptive_universe_forward_return_vs_fixed_topm')
h2_div = test_row('adaptive_universe_diversification_vs_fixed_topm')
h3 = test_row('xy_feasibility_vs_penalty_qaoa')
h4 = test_row('xy_optimality_gap_vs_penalty_qaoa')
h5 = tests.loc[tests.hypothesis.eq('H5')]

conclusion_md = f'''
### H1 — Không được hỗ trợ
XGBoost có Rank IC trung bình dương, nhưng chênh lệch so với EWMA không có ý nghĩa sau Holm (Δ = {h1.mean_difference:.4f}; CI [{h1.ci_low:.4f}, {h1.ci_high:.4f}]; p-Holm = {h1.p_value_holm:.3f}). Vì vậy chưa có bằng chứng rằng XGBoost tạo giá trị dự báo bổ sung.

### H2 — Chỉ được hỗ trợ một phần
AUR làm giảm tương quan tuyệt đối trung bình so với Top-M (Δ = {h2_div.mean_difference:.4f}; p-Holm = {h2_div.p_value_holm:.3f}), nhưng không cải thiện forward return có ý nghĩa (Δ = {h2_return.mean_difference:.4f}; p-Holm = {h2_return.p_value_holm:.3f}). Kết luận phù hợp là cải thiện đa dạng hóa, chưa chứng minh cải thiện lợi nhuận.

### H3 — Được hỗ trợ trong simulator lý tưởng
XY-QAOA/Dicke có feasibility cao hơn Penalty-QAOA (Δ = {h3.mean_difference:.4f}; CI [{h3.ci_low:.4f}, {h3.ci_high:.4f}]; p-Holm = {h3.p_value_holm:.3f}). Đây chủ yếu là xác nhận cơ chế bảo toàn cardinality, chưa phải bằng chứng trên phần cứng nhiễu.

### H4 — Được hỗ trợ có điều kiện
XY-QAOA cải thiện optimality gap so với Penalty-QAOA (Δ cải thiện = {h4.mean_difference:.4f}; p-Holm = {h4.p_value_holm:.3f}), nhưng không vượt Exact hoặc Simulated Annealing, vốn đạt gap bằng 0 trong cấu hình nhỏ.

### H5 — Không được hỗ trợ
Không có so sánh hiệu quả tài chính nào của full pipeline với các đối chứng đạt ý nghĩa sau Holm; p-Holm nhỏ nhất = {h5.p_value_holm.min():.3f}. Lợi nhuận dương của pipeline không đồng nghĩa có alpha thống kê và phần lớn gắn với mức tiền mặt cao trong các fold phòng thủ.

### H6 — Hoàn thành phân tích độ nhạy, không phải bằng chứng ưu thế
Hệ thống đã chạy lưới độ sâu, shots, seeds, cardinality, nhiễu mô phỏng và chi phí. Kết quả chỉ được suy diễn trong lưới đã khai báo; phù hợp hơn khi xem H6 là câu hỏi robustness thay vì giả thuyết nhân quả.
'''
display(Markdown(conclusion_md))

## 10. Kết luận toàn hệ thống và tải kết quả
Pipeline đã chạy end-to-end, dữ liệu giá/benchmark và ràng buộc cardinality có thể kiểm toán. Tuy nhiên nghiên cứu chưa chứng minh XGBoost vượt EWMA, chưa chứng minh danh mục vượt benchmark và chưa chứng minh quantum advantage. Lợi nhuận full pipeline cần được đọc cùng tỷ trọng tiền mặt, drawdown, Sharpe và kiểm định thống kê.

In [ ]:
download_base = Path('/content') / f"ai_quantum_results_{manifest['experiment_id']}"
archive_path = shutil.make_archive(str(download_base), 'zip', root_dir=ACTIVE_EXPERIMENT)
print('Result archive created:', archive_path, f'({Path(archive_path).stat().st_size / 1e6:.1f} MB)')
try:
    from google.colab import files
    display(Markdown(f"Nhấn biểu tượng tải xuống của Colab hoặc chạy `files.download('{archive_path}')` để lấy toàn bộ artifact."))
except ImportError:
    pass